In [1]:
import os
import json
from google import genai
from dotenv import load_dotenv
from google.genai import types
import numpy as np
from embedding_service import EmbeddingService
from vector_store import VectorStore

In [2]:
store = VectorStore("embeddings_v2.parquet")
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

## Bygger prompten för modellen

In [3]:
def build_prompt(query, results):
    context = "\n\n".join(
        [r["text"] for r in results]
    )

    prompt = f"""
Du är en AI-assistent som hjälper skeppare och värdar på More Sailing att förstå kursmaterialet på ett korrekt och professionellt sätt.

Ditt mål är att:
- Ge korrekta och fullständiga svar baserade på kursmaterialet
- Hjälpa användaren att förstå detaljer och sammanhang
- Fungera som en pålitlig kunskapskälla

Dina svar ska:
- Vara tydliga och välstrukturerade
- Innehålla tillräckligt med detaljer för att ge en korrekt förståelse
- Använda korrekt terminologi från kursmaterialet

Regler:
- Använd ENDAST informationen i kontexten nedan
- Hitta inte på information
- Om svaret inte finns i kontexten: säg "Jag vet inte baserat på kursmaterialet"

Om möjligt:
- Förklara begrepp tydligt
- Ge sammanhang så att användaren förstår varför något görs
- Knyt till praktiska situationer om det framgår i materialet

När du svarar:
- Prioritera konkreta instruktioner och regler från kursmaterialet
- Ta med viktiga åtgärder (t.ex. att informera ansvarig)
- Fokusera inte enbart på tekniska detaljer utan även beteende och ansvar

Svara i följande struktur:
1. Kort svar
2. Förklaring
3. Eventuella viktiga detaljer eller undantag

KONTEXT:
{context}

FRÅGA:
{query}
"""
    return prompt

In [4]:
service = EmbeddingService(client, types)

In [5]:
def ask(query, store, service, client, k=3):
    """
    Enkel RAG-funktion:
    - tar en fråga (query)
    - hämtar relevanta chunks
    - bygger prompt
    - skickar till LLM
    """

    # 1. Embedda fråga
    query_embedding = service.embed(query)

    # 2. Sök i VectorStore
    results = store.search(query_embedding, k=k)

    # 3. Bygg prompt
    prompt = build_prompt(query, results)

    # 4. Skicka till modell (med enkel retry)
    import time

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )
            return response.text
        except Exception as e:
            print(f"Försök {attempt+1} misslyckades: {e}")
            time.sleep(5)

    return "Kunde inte få svar från modellen just nu."

# Tester
================================================================================================
Här testar jag modellen med ett gäng frågor där framförallt de två sista frågorna är viktiga test med "embeddings_v2.parquet" mot "embeddings_v1.parquet" som chunkade på 1200 tecken och då inte kunde svara på frågan om "hur man sätter segel" men som vid senare test visade sig kunna svara på frågor som v2 inte kunde exempelvis frågan om "Vad är Zlatan Otok för ställe?" Jag kommer att återkomma till detta i evaluering_final.

In [6]:
answer = ask(
    query="Vad betyder att reva och hur ska man tänka när man revar?",
    store=store,
    service=service,
    client=client
)

print(answer)

Här är vad det betyder att reva och hur man ska tänka när man revar, baserat på kursmaterialet:

1.  **Kort svar**
    Att reva innebär att minska storleken på ett segel för att anpassa sig till förändrade vindförhållanden.

2.  **Förklaring**
    Reving av segel görs för att behålla balans i båten och upprätthålla ett jämnt tryck i masten när vindstyrkan ökar. Processen involverar en serie steg:
    *   **Förberedelse:** Innan storseglet hissas, öppna och sänk lazybagen ca 2 dm (inte så långt att den fastnar i blocken). Alla revtampar ska släppas så att de löper fritt. Det är viktigt att kontrollera hur reven fungerar på en ny båt, då antalet tampar kan variera (oftast 3 rev med 4 tampar). Alla tampar som ska användas (storfall, alla rev, storskot) ska förberedas och läggas med fyra varv runt vinschen för avlastning.
    *   **Koordination:** En person ansvarar för revet vid vinschen, och en annan för storfallet.
    *   **Utförande:**
        1.  Styrmannen lägger båten på en lämplig

In [9]:
answer = ask(
    query="Jag har problem med min båt vem ska jag ringa?",
    store=store,
    service=service,
    client=client
)

print(answer)

Det beror på vilken typ av problem du har med båten. Här är en guide baserad på kursmaterialet:

1.  **Kort svar**
    För mindre problem som du inte kan laga själv, kontakta dina kollegor och mentorer. Större skador eller problem ska rapporteras omedelbart till charterbolagets servicepersonal. Vid skador efter till exempel en tilläggning, kontakta din platsansvarig och ansvarig tekniker för din båt. Alla problem ska rapporteras till en mentor.

2.  **Förklaring**
    Som skeppare är du ansvarig för att försöka laga enklare saker som går sönder. Om du inte kan åtgärda problemet själv, följ dessa steg:

    *   **Enklare problem som du inte kan laga själv:**
        1.  Vänd dig först till avsnittet "Felsökning båt" (ej inkluderat i denna kontext).
        2.  Om problemet kvarstår, kontakta dina kollegor och dina mentorer för hjälp.
        3.  **Alla saker som går sönder ska rapporteras till en mentor.**

    *   **Större skador eller problem med båten:**
        1.  Kontakta servicep

In [10]:
answer = ask(
    query="Vad finns det att göra på Hvar och i Hvar stad?",
    store=store,
    service=service,
    client=client
)

print(answer)

Här är vad det finns att göra på Hvar och i Hvar stad, baserat på kursmaterialet:

**1. Kort svar**
På Hvar och i Hvar stad kan man utforska historiska platser som Spanska fortet, uppleva den livliga stadskärnan med ett stort utbud av restauranger och barer, besöka ateljéer, njuta av bad och strandklubbar som Hula Hula, samt utforska andra delar av ön med ankringsvikar och charmiga byar som Vrboska.

**2. Förklaring**

**I Hvar stad:**
*   **Historiska och kulturella upplevelser:**
    *   **Spanska fortet (Forticafäsningen):** Ett viktigt historiskt landmärke byggt under Venetianernas styre som vakar över staden. Det är en befästning där besökare kan uppleva dess historia och troligen njuta av utsikten.
    *   **Turistbyrån:** En av Europas äldsta, etablerad 1868.
    *   **Ateljéer:** I de trånga gränderna finns många ateljéer där lokala konstnärer ställer ut sina verk.
*   **Mat och dryck:**
    *   **Restauranger:** Ett stort utbud finns, inklusive:
        *   **Kapetana:** En lu

In [12]:
answer = ask(
    query="Hur ser rutinerna ut kring mathantering?",
    store=store,
    service=service,
    client=client
)

print(answer)

Här är rutinerna kring mathantering, baserat på kursmaterialet:

**1. Kort svar**
Rutinerna för mathantering omfattar strikt personlig hygien, noggrann hantering av råvaror, konstant upprätthållande av renlighet i köket och med redskap, korrekt temperaturkontroll för förvaring, samt tydliga instruktioner för hantering av kvarvarande matvaror vid båtbyte eller landgång. Syftet är att säkerställa matsäkerhet, undvika matförgiftning och upprätthålla en hög standard.

**2. Förklaring**

*   **Personlig hygien:**
    *   Tvätta händerna i minst 20 sekunder och använd alcogel ofta.
    *   Använd förkläde.
    *   Långt hår ska vara noga uppsatt.
    *   Inte ha långa, målade naglar eller smycken.
    *   Använd handskar vid hantering av kött och kyckling (fördel), och om du har sår på händerna. Plåster ska vara i en färg som avviker från huden och maten.

*   **Hantering av råvaror och mat:**
    *   Kontrollera att bäst före-datum inte har passerat och att råvarorna ser bra ut och luktar f

In [13]:
answer = ask(
    query="Vad ska jag göra om en gäst missköter sig?",
    store=store,
    service=service,
    client=client
)

print(answer)

Här är vad du ska göra om en gäst missköter sig, baserat på kursmaterialet:

1.  **Kort svar**
    Du ska säga ifrån och agera enligt avsnittet "Gäster som missköter sig", alltid börja med dialog och eventuellt en varning. Meddela därefter omedelbart platsansvarig eller produktionsansvarig. Om gästen inte bättrar sig och äventyrar säkerheten eller påverkar andras upplevelser negativt, har ni rätt att be gästen mönstra av vid närmaste hamn.

2.  **Förklaring**
    More Sailing värderar en god stämning ombord där alla behandlas väl och respektfullt. Om en gäst trakasserar personalen, om ni uppmärksammar att en annan gäst far illa, eller om en gäst äventyrar säkerheten ombord eller uppträder så att övriga gästers upplevelser påverkas negativt, ska ni agera enligt följande:
    *   **Säg ifrån och agera:** Gå tillväga enligt instruktionerna i avsnittet om "Gäster som missköter sig" (vilket innebär att du har rätt att be gästen mönstra av vid närmaste hamn om situationen eskalerar).
    *  

In [14]:
answer = ask(
    query="Hur ser rutinerna ut kring barn ombord??",
    store=store,
    service=service,
    client=client
)

print(answer)

Här är rutinerna kring barn ombord baserat på kursmaterialet:

1.  **Kort svar**
    Rutiner kring barn ombord fokuserar på flytvästsanvändning, tillsyn och tydliga gränser för att säkerställa deras säkerhet. Föräldrarna bär huvudansvaret, men personalen har veto i säkerhetsfrågor.

2.  **Förklaring**
    *   **Flytväst:**
        *   Barn som inte är simkunniga ska *alltid* bära flytväst.
        *   Barn ska *alltid* bära flytväst på däck, oavsett simkunnighet.
        *   För äldre barn/ungdomar under 18 år är det föräldrarnas ansvar att avgöra om flytväst ska vara på, men personalen har vetorätt. Om personalen anser att barnen bör bära flytväst (t.ex. om de rör sig mycket runt på båten eller om föräldrarna saknar sjövett), ska detta meddelas och västen sättas på.
        *   På båtar i Kroatien finns More Sailing-brandade seglarvästar som ska kontrolleras.
    *   **Tillsyn och placering:**
        *   Barn ska inte vara på däck ensamma.
        *   Barn kräver extra uppmärksamhet 

In [20]:
answer = ask(
    query="Hur hissar jag säkert upp någon I masten för att ta en cool bild?",
    store=store,
    service=service,
    client=client
)

print(answer)

För att säkert hissa upp någon i masten måste säkerheten alltid prioriteras genom noggranna förberedelser, korrekt utrustning, tydlig kommunikation och ett kontrollerat genomförande.

### Förklaring

Som skeppare ansvarar du för att instruera tydligt och säkerställa att alla moment genomförs på ett säkert sätt. Följ dessa steg:

**1. Förberedelser och utrustning:**
*   **Förhållanden:** Momentet ska om möjligt utföras under lugna förhållanden.
*   **Utrustning:** Använd en godkänd båtmansstol.
*   **Fall:** Använd två separata och oberoende fall:
    *   Ett **arbetsfall** (förslagsvis storfallet) som personen hissas i.
    *   Ett **säkerhetsfall** (exempelvis dirk/genuafallet) som fungerar som en backup.
*   **Kontroll:** Kontrollera att båda fallen är i gott skick innan användning.
*   **Fästa fall:** Båda fallen ska fästas i båtmansstolen med en korrekt slagen pålstek. **Använd aldrig schackel**, eftersom de kan ge vika och orsaka allvarliga olyckor.
*   **Sista kontroll:** Kontrol

# Jämförelse tester mot chunking v1 och v2

In [17]:
answer = ask(
    query="Hur sätter man segel?",
    store=store,
    service=service,
    client=client
)

print(answer)

För att sätta segel, vilket innebär att hissa upp seglen, behöver man först utföra vissa förberedande åtgärder, särskilt för storseglet.

1.  **Kort svar**
    Att sätta segel innebär att hissa upp seglen. För storseglet börjar man med att förbereda lazybagen och se till att alla revtampar är fria. Alla relevanta tampar ska också förberedas vid vinscharna.

2.  **Förklaring**
    Innan storseglet hissas upp ska följande steg genomföras:
    *   **Öppna och sänk lazybagen:** Öppna och sänk ner lazybagen med cirka 2 dm. Se till att den inte hänger så långt ner att den riskerar att fastna eller dras sönder av skotets block.
    *   **Släpp på alla rev:** Innan seglet hissas, se till att alla revtampar är släppta så att de kan löpa fritt och inte hindrar seglet från att hissas upp fullt ut.
    *   **Förbered tampar:** Förbered alla tampar som ska användas, vilket inkluderar storfallet, alla revtampar och storskotet. Lägg dem med fyra varv runt respektive vinsch för att säkerställa god avl

In [18]:
store = VectorStore("embeddings_v1.parquet")

In [19]:
answer = ask(
    query="Hur sätter man segel?",
    store=store,
    service=service,
    client=client
)

print(answer)

Jag vet inte baserat på kursmaterialet. Kursmaterialet beskriver terminologi och meddelar att en förklaring av hur man sätter segel kommer senare i texten, men de faktiska stegen för att sätta segel finns inte i den angivna kontexten.


In [22]:
answer = ask(
    query="Vad är Zlatan otok för ställe?",
    store=store,
    service=service,
    client=client
)

print(answer)

**1. Kort svar**
Zlatan Otok är en av Kroatiens bästa vingårdar, som har byggt en egen hamn för att ta emot besökare och erbjuda middag samt vinprovning.

**2. Förklaring**
På Zlatan Otok kan skeppare och värdar förtöja i vingårdens speciellt byggda hamn. Syftet med hamnen är att ge besökare möjlighet att uppleva vingårdens verksamhet. Istället för att besöka den faktiska vingården har de en utställningsmodell av en vinkällare under restaurangen. Här kan gästerna njuta av middag och vinprovning.

**3. Eventuella viktiga detaljer eller undantag**
*   **Bokning:** Platserna i hamnen är begränsade, så det är viktigt att boka i god tid.
*   **Vinprovning och middag:** Det är bra att direkt vid ankomst prata med personalen för att bestämma hur och när man vill ha middag och vinprovning. Vinprovningen tar vanligtvis 1-1,5 timmar, beroende på hur många viner som provas och om man vill ha till exempel ost eller charktallrik till.
*   **Aktiviteter:** För de äventyrslystna finns en stor grotta 

In [23]:
store = VectorStore("embeddings_v2.parquet")

In [24]:
answer = ask(
    query="Vad är Zlatan otok för ställe?",
    store=store,
    service=service,
    client=client
)

print(answer)

1.  **Kort svar**
    "Zlatan otok" refererar till ön Zlarin, som på grund av sin skönhet kallas för "gyllene ön" i kursmaterialet.

2.  **Förklaring**
    Zlarin är en ö som ligger vid inloppet till Krka-floden. Den är känd för sin skönhet och kallas därför för "gyllene ön", "gröna ön" och "korallön". På öns nordsida finns en gammal fiskeby med ett fåtal restauranger i en gemytlig miljö. Det finns även en korallverkstad som bearbetar korall för hand och säljer smycken.

3.  **Viktiga detaljer**
    *   **Lämplighet:** Zlarin är ett bra alternativ för sydliga vindar.
    *   **Hamnen:** Hamnen är liten, så det är viktigt att komma i tid för att vara säker på att få plats, annars riskerar den att bli full.
    *   **Användning:** Den är klassad som både natthamn (Lila/svart) och lunchstopp (Lila/grön).
